# Problem set 8: Bagging, random forests and boosting

**STA 363 | Posted November 13 | Due Friday, November 20**

**Getting started.** Click **File > Save a copy in Drive**, then work in your copy. Run the setup cell below before anything else.

**Turning it in.** When you are done, click **Share** in the top right, set **General access** to *Anyone with the link*, then **Copy link** and paste that link into Canvas.

---

If you work with classmates, please indicate that here. Additionally
please cite all other sources including AI.

*Type your answer here.*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingRegressor,
                              GradientBoostingClassifier)
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import confusion_matrix

DATA = "https://raw.githubusercontent.com/LucyMcGowan/sta-363-f26/main/data/"

## 1.

Suppose we draw ten bootstrap training sets from a data set with two classes,
red and green, and grow a classification tree on each one. For a new
observation $x_0$, the ten trees estimate $P(\text{Red} \mid X = x_0)$ as

$$0.1,\ 0.15,\ 0.2,\ 0.2,\ 0.55,\ 0.6,\ 0.6,\ 0.65,\ 0.7,\ 0.75.$$

There are two common ways to combine these into one prediction. A **majority
vote** lets each tree predict Red when its estimate is above 0.5, then predicts
the class with the most votes. **Averaging the probabilities** predicts Red when
the mean of the ten estimates is above 0.5.

**a.** By hand, which class does the majority vote predict for $x_0$?

*Type your answer here.*

**b.** By hand, which class does averaging the probabilities predict for $x_0$?

*Type your answer here.*

**c.** Explain why the two approaches disagree for $x_0$. Hint: compare how far the estimates on each side of 0.5 are from 0.5.

*Type your answer here.*

**d.** Bagged trees are grown deep and are not pruned. Suppose each tree is grown until every leaf holds training observations from a single class. What values can each tree's estimate take? Explain why the majority vote and the averaged probability then always predict the same class.

*Type your answer here.*

`RandomForestClassifier` with `max_features=None` is bagging for
classification. `min_samples_leaf` sets the fewest training observations a leaf
may hold, so `min_samples_leaf=10` grows shallower trees whose leaves often mix
the two classes. The code below reads in `Carseats`, where `High` is Yes when `Sales` is above 8, and splits the 400 stores in
half. `tree_probs` returns each tree's estimate of $P(\text{High} = \text{Yes})$,
with one row per tree and one column per store.

In [ ]:
Carseats = pd.read_csv(DATA + "Carseats.csv")
X_seats = pd.get_dummies(Carseats.drop(columns=["Sales"]), drop_first=True).astype(float)
High = np.where(Carseats["Sales"] > 8, "Yes", "No")
seats_train, seats_test, High_train, High_test = train_test_split(
    X_seats, High, test_size=0.5, random_state=363
)


def tree_probs(forest, X_new):
    return np.array([tree.predict_proba(X_new.to_numpy())[:, 1]
                     for tree in forest.estimators_])

**e.** Fill in the blanks below. Each forest has 501 trees, an odd number, so the vote cannot tie. For each random forest, how many distinct estimates do the trees give, and for how many of the 200 test stores do the two rules disagree? Does this match part d? Which rule does `forest.predict` follow?

In [ ]:
rows = []
for leaf in [1, 10]:
    forest = RandomForestClassifier(n_estimators=501, max_features=None,
                                    min_samples_leaf=leaf, random_state=363)
    forest.fit(seats_train, High_train)
    probs = tree_probs(forest, seats_test)
    # Yes when more than half of the 501 trees estimate P(Yes) above 0.5
    vote = np.where(____, "Yes", "No")
    # Yes when the mean of the 501 estimates is above 0.5
    average = np.where(____, "Yes", "No")
    rows.append({
        "min_samples_leaf": leaf,
        "distinct estimates": len(np.unique(probs)),
        "stores where the rules disagree": int((vote != average).sum()),
        "predict matches vote": bool((forest.predict(seats_test) == vote).all()),
        "predict matches average": bool((forest.predict(seats_test) == average).all()),
    })

pd.DataFrame(rows)

*Type your answer here.*

## 2.

ISLP Section 8.2.3 states that boosting with stumps (trees with $d = 1$ split)
gives an **additive model**, a model of the form

$$f(X) = \sum_{j=1}^{p} f_j(X_j),$$

where each $f_j$ depends only on the single predictor $X_j$. The boosted model
from lecture is

$$\hat{f}(x) = \sum_{b=1}^{B}\lambda\hat{f}^b(x).$$

**a.** Explain why boosting with stumps gives an additive model. Hint: what does the prediction of a single stump depend on? Group the $B$ shrunken stumps by the predictor each one splits on.

*Type your answer here.*

The loop below is the boosting loop from lecture, fit to `Sales` in `Carseats`
with the 11 columns of `X_seats` from Question 1. The fit
starts at the mean of `Sales` rather than at 0, as `GradientBoostingRegressor`
does, so the residuals start at `Sales` minus its mean. It also uses
$\lambda = 0.1$. `stump.tree_.feature[0]` is the column number of the
predictor used at a tree's first split, which for a stump is its only split,
and `split_on` records the name of that predictor for every stump. The table at
the end counts how many of the first 200 stumps and how many of the last 200
split on each predictor.

**b.** Fill in the blanks below to run the loop. Which predictors do the first 200 stumps split on most often, and which do the last 200 split on most often? What do you think the late stumps on `Population` are fitting, and what does that suggest about using 1,000 stumps here?

In [ ]:
sales = Carseats["Sales"]
lam, B = 0.1, 1000
r = (sales - sales.mean()).to_numpy()
stumps = []
split_on = []
for b in range(B):
    # a tree with d = 1 split, fit to the current residuals
    stump = DecisionTreeRegressor(max_leaf_nodes=____, random_state=363)
    stump.fit(X_seats, ____)
    # subtract the shrunken stump from the residuals
    r = ____
    stumps.append(stump)
    split_on.append(X_seats.columns[stump.tree_.feature[0]])

f_hat = sales.mean() + sum(lam * stump.predict(X_seats) for stump in stumps)
pd.DataFrame({
    "first 200": pd.Series(split_on[:200]).value_counts(),
    "last 200": pd.Series(split_on[-200:]).value_counts(),
}).fillna(0).astype(int)

*Type your answer here.*

**c.** `f_j` below adds up the shrunken stumps that split on one column, which is the $f_j$ from part a. Fill in the blanks. What does the last line check, and how does its output confirm part a?

In [ ]:
def f_j(column, X_new):
    total = np.zeros(len(X_new))
    for stump, name in zip(stumps, split_on):
        # keep only the stumps that split on this column
        if ____:
            total = total + ____
    return total


parts = pd.DataFrame({col: f_j(col, X_seats) for col in X_seats.columns})
np.allclose(sales.mean() + parts.sum(axis=1), f_hat)

*Type your answer here.*

The code below plots $f_{\text{Price}}$ across the observed prices. `grid`
copies the first store 200 times and then replaces its price.

**d.** Why would copying a different store into `grid` give the same curve? By about how much does the boosted model lower predicted sales when the price rises from 100 to 140? Compare with 40 times the coefficient on `Price` from `LinearRegression().fit(X_seats, sales)`.

In [ ]:
grid = X_seats.iloc[[0] * 200].reset_index(drop=True)
grid["Price"] = np.linspace(X_seats["Price"].min(), X_seats["Price"].max(), 200)

plt.plot(grid["Price"], f_j("Price", grid), color="#d5008f")
plt.xlabel("Price")
plt.ylabel(r"$f_{\mathrm{Price}}$")
plt.show()

*Type your answer here.*

**e.** With $d = 2$, a tree can split on `ShelveLoc_Good` and then split only the good-shelf stores on `Price`, giving three leaves. Explain why its prediction cannot be written as a function of `ShelveLoc_Good` plus a function of `Price` unless two leaves predict the same sales.

*Type your answer here.*

In application exercise 15, logistic regression beat the classification trees
on `High`. For part f, set
`cv = KFold(n_splits=10, shuffle=True, random_state=363)` and give each
`GradientBoostingRegressor` the argument `random_state=363`.

**f.** Use `cross_val_score` and `cv` to find the CV MSE of `LinearRegression()` and of boosting with 1,000 trees and $\lambda = 0.1$, for $d = 1$ and for $d = 2$. Rank the three. Using parts d and e, what does the ranking suggest about how `Sales` depends on the predictors?

*Type your answer here.*

## 3.

The `Caravan` data includes whether each of 5,822
customers bought caravan insurance, and 348 of them did. Below, the first 1,000
customers are the training set and the remaining 4,822 form the test set. Names starting
with P give a customer's contribution to one type of insurance policy and names
starting with A give the number of policies of that type, so `PPERSAUT` is the
contribution to car policies.

In [ ]:
Caravan = pd.read_csv(DATA + "Caravan.csv")
y_car = (Caravan["Purchase"] == "Yes").astype(int)
X_car = Caravan.drop(columns=["Purchase"])
X_train, X_test = X_car.iloc[:1000], X_car.iloc[1000:]
y_train, y_test = y_car.iloc[:1000], y_car.iloc[1000:]
int(y_train.sum()), int(y_test.sum())

`GradientBoostingClassifier` boosts trees for a two-class response. Each tree
is fit to the gap between the 0/1 response and the current estimated
probability, and the shrunken trees add up on the log-odds scale, the same
scale as logistic regression. `learning_rate` is $\lambda$ and
`max_leaf_nodes` is $d + 1$, as in `GradientBoostingRegressor`, and
`.predict_proba()` returns estimated probabilities.

**a.** Fit `GradientBoostingClassifier` to the training set with 1,000 stumps, $\lambda = 0.01$ and `random_state=363`, and save it as `boost`. Which five predictors have the largest `feature_importances_`? How many of the 85 have an importance of exactly 0? How can that happen?

*Type your answer here.*

Your book predicts a purchase when the estimated probability of a purchase is above
20%. A customer mailed a brochure at random buys about 6% of the time.

**b.** Use `boost` to estimate the probability of a purchase for each test customer, save the predictions above 20% as `flagged`, and form `confusion_matrix(y_test, flagged)`. What fraction of the flagged customers purchase, compared with 6%? How many would a threshold of 50% flag?

*Type your answer here.*

**c.** Fill in the blanks below to repeat part b for logistic regression and a random forest. Which model has the highest precision at 20%, and how many customers does each one flag?

In [ ]:
models = {
    "boosting": boost,
    "logistic regression": make_pipeline(StandardScaler(),
                                         LogisticRegression(max_iter=10000)),
    "random forest": RandomForestClassifier(n_estimators=500, max_features="sqrt",
                                            oob_score=True, random_state=363),
}
probs = {}
rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    probs[name] = model.predict_proba(X_test)[:, 1]
    # True for each test customer predicted to purchase
    flagged = ____
    rows.append({
        "model": name,
        "flagged": int(flagged.sum()),
        "purchased": int(y_test[flagged].sum()),
        # fraction of the flagged customers who purchased
        "precision": ____,
    })

pd.DataFrame(rows).round(3)

*Type your answer here.*

The forest's `oob_score_` is its OOB accuracy: each training customer is
predicted by the trees that left it out, with a threshold of 50%.

**d.** Compare `models["random forest"].oob_score_` with the accuracy of predicting no purchase for every training customer. Why is OOB accuracy a poor guide to how useful this random forest is for choosing who gets a brochure?

*Type your answer here.*

The three models flag different numbers of customers, so their precisions in
part c compare different mailing sizes.

**e.** Suppose the company can mail as many brochures as boosting flagged in part b. Fill in the blanks below to find each model's precision among that many test customers with the highest estimated probabilities. Which model would you use to choose who gets a brochure?

In [ ]:
# the number of test customers boosting flagged in part b
k = ____
for name, p in probs.items():
    # np.argsort(-p) lists the test customers from the largest p to the smallest
    top = np.argsort(-p)[:k]
    print(name, round(float(____), 3))

*Type your answer here.*

*Data from James, G., Witten, D., Hastie, T., Tibshirani, R., and Taylor, J.
(2023). An Introduction to Statistical Learning with Applications in Python.
Springer. Questions 1 to 3 are adapted from Chapter 8 exercises 5, 2 and 11.*